# Phase 7 — Machine-Learning Models (XGBoost + GARCH-X)

Run this notebook on **Colab Pro** with the model matrix and the outputs
in your Google Drive folder `WarSignalsThesis_Data/`.

**What it does:**
1. Mounts Drive and clones the repo (with the `git pull` fix to avoid stale clones)
2. Verifies the model matrix exists in Drive
3. Runs TS-CV grid search for XGBoost (Phase 7.3) → `xgb_best_params.csv`
4. Runs the OOS horse race (Phase 7.4) → `phase7_benchmark.csv` etc.
5. Computes SHAP and writes `fig17_shap_summary_*.png`
6. Verifies all outputs are on Drive

**Important:** rclone is local-only. After this notebook finishes, switch to your
local laptop and run `rclone copy --update --progress` to pull the outputs back.

## 1. Mount Drive, clone repo, sanity check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/WarSignalsThesis_Data')
REPO_DIR = Path('/content/WarSignalsThesis')

# Clone if needed, then ALWAYS git pull (per colab_notebook_lesson)
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/katerynavalenia/WarSignalsThesis'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', 'origin', 'main'], check=True)

# Sanity check: ensure recent script exists
expected = REPO_DIR / 'scripts' / 'phase7_run_ml.py'
if not expected.exists():
    raise FileNotFoundError(f'{expected} missing after pull')
print(f'✓ Repo at {REPO_DIR}, {expected.name} present')
print(f'✓ Drive data at {DRIVE_ROOT}')

## 2. Install dependencies

In [ ]:
# xgboost and shap were already in requirements.txt for Phase 7
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost>=2.0', 'shap>=0.44'], check=True)
import xgboost, shap
print(f'xgboost {xgboost.__version__}, shap {shap.__version__}')

## 3. Verify model matrix in Drive

In [ ]:
import pandas as pd
MM_PATH = DRIVE_ROOT / 'data' / 'processed' / 'model_matrix.parquet'
if not MM_PATH.exists():
    raise FileNotFoundError(
        f'{MM_PATH} missing. Did you push it to Drive? (Phase 7.0)\n'
        f'Local command: rclone copy --update --progress data/processed/ '
        f'gdrive:WarSignalsThesis_Data/data/processed/'
    )
mm = pd.read_parquet(MM_PATH)
print(f'✓ Model matrix: {mm.shape[0]} rows × {mm.shape[1]} cols')
print(f'  Date range: {mm["date"].min()} → {mm["date"].max()}')

## 4. Run grid search (tuning) — ~110 min on Colab Pro

Writes `xgb_best_params.csv` directly to Drive at `outputs/model_objects/`.

**Skip this cell** if you have already run tuning and the CSV is in Drive.

In [ ]:
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / 'tables'
MODEL_OBJECTS_DIR = DRIVE_ROOT / 'outputs' / 'model_objects'
FIG_DIR = DRIVE_ROOT / 'outputs' / 'figures'
TUNED_PARAMS_CSV = MODEL_OBJECTS_DIR / 'xgb_best_params.csv'

for d in (OUTPUT_DIR, MODEL_OBJECTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Run the tuner. Use --quick for a 4-config × 2-fold smoke test (~2 min).
cmd = [
    sys.executable, '-m', 'src.models.ml_tuning',
    '--data-path', str(MM_PATH),
    '--output', str(TUNED_PARAMS_CSV),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print(f'✓ Wrote {TUNED_PARAMS_CSV}')

## 5. Run OOS horse race (XGBoost returns + GARCH-X vol) — ~30 min on Colab Pro

In [ ]:
cmd = [
    sys.executable, 'scripts/phase7_run_ml.py',
    '--data-path', str(MM_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--tuned-params', str(TUNED_PARAMS_CSV),
    '--garch-x-info-set', 'F',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('✓ Phase 7 horse race done.')

## 6. Verify outputs

In [ ]:
import subprocess
print('=== outputs/tables/phase7_* ===')
subprocess.run(['ls', '-lh', str(OUTPUT_DIR)], check=False)
print('=== outputs/figures/fig17* ===')
subprocess.run(['ls', '-lh', str(FIG_DIR)], check=False)
print('=== outputs/model_objects/ ===')
subprocess.run(['ls', '-lh', str(MODEL_OBJECTS_DIR)], check=False)

benchmark = pd.read_csv(OUTPUT_DIR / 'phase7_benchmark.csv')
print(f'\n✓ phase7_benchmark.csv: {len(benchmark)} rows')
print(f'  Models: {sorted(benchmark["model"].unique())}')
print(f'  Info sets: {sorted(benchmark["info_set"].unique())}')
print('\n  XGBoost MAE on F (h=1):')
    
xgb_f = benchmark[(benchmark['model'] == 'xgboost') & (benchmark['info_set'] == 'F') & (benchmark['horizon'] == 1)]
print(xgb_f[['target', 'MAE', 'RMSE', 'dir_acc']].to_string(index=False))

## 7. Next step: pull outputs to local

**On your local laptop** (NOT in this notebook — rclone is local-only per
[user memory](docs/data_sharing.md)):

```bash
cd ~/Desktop/katya/WarSignalsThesis
source .venv/bin/activate

# Pull results from Drive
rclone copy --update --progress \
  gdrive:WarSignalsThesis_Data/outputs/tables/phase7_* \
  outputs/tables/
rclone copy --update --progress \
  gdrive:WarSignalsThesis_Data/outputs/model_objects/ \
  outputs/model_objects/
rclone copy --update --progress \
  gdrive:WarSignalsThesis_Data/outputs/figures/fig17* \
  outputs/figures/

# Verify
ls -lh outputs/tables/phase7_* outputs/figures/fig17*

# Commit to git
git add -A
git commit -m 'Phase 7: XGBoost returns + GARCH-X vol + SHAP'
git push origin main
```